# 13 — Zero-Shot LLM Classifier

## Objective

Give a free, open-source LLM your **annotation guidelines** (no training examples) and let it label every passage.
Then compare its labels with yours.

**Why:** notebook 11 showed that word-based models (TF-IDF) stay at chance on the strict labels. They detect *topic*
(music, books, emotion), not *function*. An LLM reads for meaning, so the question is: **can it apply your rules?**

## Before you start (Colab)

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run the cells in order. The first model download takes a few minutes (~5 GB).
3. The full run takes roughly **10–20 minutes** for 79 passages with the 7B model (faster with 3B).

## Important caveats

- The LLM's labels are **not ground truth**. It's a consistent "second reader" to compare with.
- Agreement between you and the LLM is **not** human inter-annotator agreement. You still need a second human (Dr. Neha) for that.
- The model has probably seen *Frankenstein* during training, which may help it. A McEwan test would be harder.
- If you labelled Batch 03 after seeing Claude's suggestions, say so when you report results.

## 1. Settings

In [2]:
MODEL_NAME   = "Qwen/Qwen2.5-7B-Instruct"   # best quality on a T4 (loaded in 4-bit)
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # faster, less memory: use if the 7B is too slow or crashes

DATA_FILE    = "frankenstein_training_dataset_v4_strict.csv"
RULE_VERSION = "v1.1_strict"   # or "v1.0_lenient" to see how much the rule wording matters
SMOKE_TEST_N = 3               # passages to try before the full run
MAX_NEW_TOKENS = 220
DRY_RUN = False                # True = fake answers, to test the notebook without a GPU

## 2. Install and check the GPU

In [3]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not DRY_RUN:
    !pip install -q -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.6 MB/s eta 0:00:00


In [4]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score, precision_score, recall_score)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

if not DRY_RUN:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, "
                           "then Runtime > Restart session and run again.")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("DRY RUN mode: no GPU needed, answers will be fake.")

GPU: Tesla T4


## 3. Load your labelled dataset

In [5]:
def find_file(name):
    for p in [name, f"data/processed/{name}", f"../data/processed/{name}"]:
        if Path(p).exists():
            return p
    if IN_COLAB:
        from google.colab import files
        print(f"Please upload {name}")
        files.upload()
        return name
    raise FileNotFoundError(f"Could not find {name}. Put it next to this notebook.")


df = pd.read_csv(find_file(DATA_FILE))
df = df.dropna(subset=["text", "therapeutic_label"]).reset_index(drop=True)
df["therapeutic_label"] = df["therapeutic_label"].astype(int)
for col in ["selection_method", "target", "mechanism", "annotator_notes"]:
    if col not in df:
        df[col] = ""
df[["selection_method", "target"]] = df[["selection_method", "target"]].fillna("")

print("Passages:", len(df))
print(df["therapeutic_label"].value_counts().rename({0: "not therapeutic (0)", 1: "therapeutic (1)"}).to_string())

Please upload frankenstein_training_dataset_v4_strict.csv


Saving frankenstein_training_dataset_v4_strict.csv to frankenstein_training_dataset_v4_strict.csv
Passages: 79
therapeutic_label
not therapeutic (0)    63
therapeutic (1)        16


## 4. The prompt (your guidelines)

This is everything the LLM knows about the task. **Edit the rules here** if your guidelines change, then re-run.

In [6]:
MECHANISMS = ["Narrative Reconstruction", "Emotional Catharsis", "Scriptotherapy",
              "Poetry-mediated Processing", "Music-mediated Processing"]

RULES = {
    "v1.1_strict": """LABEL RULES (v1.1, strict):
- Label 1 ONLY if the passage shows emotional processing that reaches some outcome: new understanding, release of emotion, calm, or a changed perspective.
- Reflection or memory that ends in despair, rage or revenge, without any resolution, is 0.
- Receiving support or comfort is 0, unless the character actually releases emotion or is changed by it.
- Choosing to tell one's painful story to someone can count as Narrative Reconstruction.
- Emotion alone, trauma alone, or memory alone is 0.
- Judge the FUNCTION of the passage, not its topic. Mentions of music, poetry, writing, letters or books do NOT make a passage therapeutic by themselves.""",

    "v1.0_lenient": """LABEL RULES (v1.0, lenient):
- Label 1 if the passage shows emotional processing, meaning-making, self-reflection leading to understanding, narrative reconstruction, emotional release through expression, or supportive interactions that contribute to emotional regulation.
- An attempt at processing counts, even if the character remains in distress.
- Label 0 for traumatic events only, sadness without processing, physical descriptions, ordinary conversations, or emotional reactions without transformation.
- Emotion alone, trauma alone, or memory alone is 0.""",
}

MECH_DEFS = """MECHANISMS (choose exactly one if the label is 1):
- Narrative Reconstruction: a character reorganizes past experiences into a meaningful personal narrative.
- Emotional Catharsis: emotional expression or interpersonal exchange that allows release or regulation of distress.
- Scriptotherapy: writing or storytelling used as a means of emotional processing.
- Poetry-mediated Processing: poetic expression used for emotional understanding or transformation.
- Music-mediated Processing: music or artistic sound experiences used for emotional regulation or reflection.

INTENSITY (0 if the label is 0): 1 very weak, 2 weak, 3 moderate, 4 strong, 5 very strong."""

SYSTEM_PROMPT = (
    "You are assisting a literary scholar in the medical humanities. You annotate passages from "
    "Mary Shelley's Frankenstein for representations of therapeutic processes. You are not diagnosing "
    "anyone; you identify how the narrative represents emotional processing.\n\n"
    + RULES[RULE_VERSION] + "\n\n" + MECH_DEFS + "\n\n"
    "Respond with ONLY a JSON object, no other text, in this format:\n"
    '{"reason": "<one or two sentences>", "evidence": "<short phrase from the passage, max 15 words>", '
    '"therapeutic_label": <0 or 1>, "mechanism": "<one mechanism name, or none>", "intensity": <0-5>}'
)


def make_user_prompt(pid, text):
    return f'Passage {pid}:\n"""\n{text}\n"""\n\nAnnotate this passage. Reply with the JSON object only.'


print(SYSTEM_PROMPT)

You are assisting a literary scholar in the medical humanities. You annotate passages from Mary Shelley's Frankenstein for representations of therapeutic processes. You are not diagnosing anyone; you identify how the narrative represents emotional processing.

LABEL RULES (v1.1, strict):
- Label 1 ONLY if the passage shows emotional processing that reaches some outcome: new understanding, release of emotion, calm, or a changed perspective.
- Reflection or memory that ends in despair, rage or revenge, without any resolution, is 0.
- Receiving support or comfort is 0, unless the character actually releases emotion or is changed by it.
- Choosing to tell one's painful story to someone can count as Narrative Reconstruction.
- Emotion alone, trauma alone, or memory alone is 0.
- Judge the FUNCTION of the passage, not its topic. Mentions of music, poetry, writing, letters or books do NOT make a passage therapeutic by themselves.

MECHANISMS (choose exactly one if the label is 1):
- Narrative

## 5. Load the model

In [7]:
if DRY_RUN:
    def ask_llm(system, user):
        # FAKE answers for testing the notebook only. This is NOT a model.
        t = user.lower()
        lab = int(any(w in t for w in ["understand", "relief", "calm", "consolation"]))
        return json.dumps({"reason": "dry run", "evidence": "", "therapeutic_label": lab,
                           "mechanism": "Narrative Reconstruction" if lab else "none",
                           "intensity": 2 if lab else 0})
    print("DRY RUN: fake answers, no model loaded")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    big = any(s in MODEL_NAME for s in ["7B", "8B"])
    quant = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16) if big else None)
    load_kwargs = dict(device_map="auto", quantization_config=quant)
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, **load_kwargs)
    except TypeError:  # older transformers versions
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, **load_kwargs)
    model.eval()

    def ask_llm(system, user):
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("Loaded", MODEL_NAME)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct


## 6. Reading the answer

The model replies in JSON. This turns the reply into clean columns, even if the model adds extra text
or code fences. If a reply can't be read, it's marked `parse_ok = False` and left out of the scores.

In [8]:
MECH_KEYS = {"narrative": "Narrative Reconstruction", "catharsis": "Emotional Catharsis",
             "scripto": "Scriptotherapy", "poetry": "Poetry-mediated Processing",
             "music": "Music-mediated Processing"}


def parse_answer(raw):
    raw = str(raw)
    data = {}
    m = re.search(r"\{.*\}", raw, re.S)
    if m:
        try:
            data = json.loads(m.group(0))
        except json.JSONDecodeError:
            data = {}
    label = data.get("therapeutic_label")
    if label is None:  # fallback: find the label even if the JSON is broken
        lm = re.search(r'therapeutic_label"?\s*[:=]\s*"?([01])', raw)
        label = lm.group(1) if lm else None
    try:
        label = int(label) if label is not None else None
    except (TypeError, ValueError):
        label = None
    mech_raw = str(data.get("mechanism", "") or "").lower()
    mech = next((v for k, v in MECH_KEYS.items() if k in mech_raw), "")
    try:
        intensity = int(data.get("intensity", 0))
    except (TypeError, ValueError):
        intensity = 0
    if label == 0:
        mech, intensity = "", 0
    return {"llm_label": label, "llm_mechanism": mech, "llm_intensity": intensity,
            "llm_reason": str(data.get("reason", "")).strip(),
            "llm_evidence": str(data.get("evidence", "")).strip(),
            "parse_ok": label in (0, 1)}


print(parse_answer('```json\n{"reason": "test", "therapeutic_label": 1, "mechanism": "Music-mediated", "intensity": 2}\n```'))

{'llm_label': 1, 'llm_mechanism': 'Music-mediated Processing', 'llm_intensity': 2, 'llm_reason': 'test', 'llm_evidence': '', 'parse_ok': True}


## 7. Smoke test on a few passages

Check that the answers look sensible before the full run.

In [9]:
for _, r in df.head(SMOKE_TEST_N).iterrows():
    t0 = time.time()
    raw = ask_llm(SYSTEM_PROMPT, make_user_prompt(r["passage_id"], r["text"]))
    p = parse_answer(raw)
    print(f"--- Passage {r['passage_id']}  ({time.time() - t0:.1f}s)  your label: {r['therapeutic_label']}")
    print("raw   :", raw.strip()[:400])
    print("parsed:", {k: p[k] for k in ['llm_label', 'llm_mechanism', 'llm_intensity', 'parse_ok']}, "\n")

--- Passage 1  (5.2s)  your label: 0
raw   : {"reason": "No emotional processing or outcome shown.", "evidence": "", "therapeutic_label": 0, "mechanism": "none", "intensity": 0}
parsed: {'llm_label': 0, 'llm_mechanism': '', 'llm_intensity': 0, 'parse_ok': True} 

--- Passage 2  (3.7s)  your label: 1
raw   : {"reason": "Victor reflects on his past, finding peace and purpose.", "evidence": "These reflections have dispelled the agitation", "therapeutic_label": 1, "mechanism": "Narrative Reconstruction", "intensity": 3}
parsed: {'llm_label': 1, 'llm_mechanism': 'Narrative Reconstruction', 'llm_intensity': 3, 'parse_ok': True} 

--- Passage 5  (4.8s)  your label: 0
raw   : {"reason": "Victor expresses his desire for a friend to share his feelings and thoughts.", "evidence": "I have no one near me, gentle yet courageous", "therapeutic_label": 1, "mechanism": "Narrative Reconstruction", "intensity": 3}
parsed: {'llm_label': 1, 'llm_mechanism': 'Narrative Reconstruction', 'llm_intensity': 3, '

## 8. Full run

Answers are saved after every passage. If Colab disconnects or you stop the cell,
just run it again: it skips passages that are already done (as long as the session's files still exist).

In [10]:
TAG = MODEL_NAME.split("/")[-1] + "_" + RULE_VERSION + ("_DRYRUN" if DRY_RUN else "")
RAW_FILE = RESULTS_DIR / f"llm_raw_{TAG}.csv"

rows = pd.read_csv(RAW_FILE).to_dict("records") if RAW_FILE.exists() else []
done_ids = {r["passage_id"] for r in rows}
todo = df[~df["passage_id"].isin(done_ids)]
print(f"Already done: {len(done_ids)} | To do: {len(todo)}")

start = time.time()
for k, (_, r) in enumerate(todo.iterrows(), start=1):
    raw = ask_llm(SYSTEM_PROMPT, make_user_prompt(r["passage_id"], r["text"]))
    rows.append({"passage_id": r["passage_id"], "raw": raw})
    pd.DataFrame(rows).to_csv(RAW_FILE, index=False)
    if k % 5 == 0 or k == len(todo):
        print(f"{k}/{len(todo)} done  ({(time.time() - start) / k:.1f}s per passage)")

raw_df = pd.DataFrame(rows)
preds = pd.concat([raw_df[["passage_id"]], raw_df["raw"].apply(parse_answer).apply(pd.Series)], axis=1)
res = df.merge(preds, on="passage_id", how="inner")
print(f"\nParsed OK: {res['parse_ok'].sum()} of {len(res)}")
if (~res["parse_ok"]).any():
    print("Could not read the answer for:", res.loc[~res["parse_ok"], "passage_id"].tolist())

Already done: 0 | To do: 79
5/79 done  (4.2s per passage)
10/79 done  (4.1s per passage)
15/79 done  (4.1s per passage)
20/79 done  (4.4s per passage)
25/79 done  (4.4s per passage)
30/79 done  (4.5s per passage)
35/79 done  (4.5s per passage)
40/79 done  (4.6s per passage)
45/79 done  (4.6s per passage)
50/79 done  (4.7s per passage)
55/79 done  (4.8s per passage)
60/79 done  (4.8s per passage)
65/79 done  (4.8s per passage)
70/79 done  (4.8s per passage)
75/79 done  (4.8s per passage)
79/79 done  (4.8s per passage)

Parsed OK: 79 of 79


## 9. How well does the LLM agree with you?

In [11]:
ev = res[res["parse_ok"]].copy()
y_true = ev["therapeutic_label"].astype(int)
y_pred = ev["llm_label"].astype(int)


def scores(y_t, y_p):
    return {"balanced_accuracy": balanced_accuracy_score(y_t, y_p),
            "precision": precision_score(y_t, y_p, zero_division=0),
            "recall": recall_score(y_t, y_p, zero_division=0),
            "f1": f1_score(y_t, y_p, zero_division=0),
            "cohen_kappa": cohen_kappa_score(y_t, y_p)}


table = pd.DataFrame({
    f"LLM ({RULE_VERSION})": scores(y_true, y_pred),
    "Always say 1": scores(y_true, np.ones_like(y_true)),
    "Always say 0": scores(y_true, np.zeros_like(y_true)),
}).T.round(3)
display(table)

cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=[0, 1]),
                  index=["you: 0", "you: 1"], columns=["LLM: 0", "LLM: 1"])
display(cm)
print("Reminder from notebook 11: on the strict labels, TF-IDF models were at chance (balanced accuracy ~0.50).")

,balanced_accuracy,precision,recall,f1,cohen_kappa
LLM (v1.1_strict),0.55,0.226,0.75,0.348,0.053
Always say 1,0.50,0.203,1.00,0.337,0.000
Always say 0,0.50,0.000,0.00,0.000,0.000


,LLM: 0,LLM: 1
you: 0,22,41
you: 1,4,12


Reminder from notebook 11: on the strict labels, TF-IDF models were at chance (balanced accuracy ~0.50).


### Was it fooled by the hard negatives?

In notebook 11 (5b), the TF-IDF model called most music, poetry and reading passages therapeutic.
Here's the same breakdown for the LLM.

In [12]:
ev["false_pos"] = (ev["llm_label"] == 1) & (ev["therapeutic_label"] == 0)
ev["false_neg"] = (ev["llm_label"] == 0) & (ev["therapeutic_label"] == 1)
group_col = ev["target"].where(ev["target"] != "", ev["selection_method"])
breakdown = (ev.assign(group=group_col)
               .groupby("group")
               .agg(n=("passage_id", "size"), your_1s=("therapeutic_label", "sum"),
                    llm_1s=("llm_label", "sum"), false_pos=("false_pos", "sum"),
                    false_neg=("false_neg", "sum"))
               .sort_values("false_pos", ascending=False))
display(breakdown)

,n,your_1s,llm_1s,false_pos,false_neg
group,,,,,
pilot,39,13,26,16,3
writing,6,1,5,5,1
support,5,0,4,4,0
poetry,6,1,5,4,0
music,6,0,3,3,0
random,4,0,3,3,0
reading,4,1,3,2,0
reflection_despair,5,0,2,2,0
walton_letters,4,0,2,2,0


### Mechanisms: when you both said 1, did you pick the same mechanism?

In [13]:
both = ev[(ev["therapeutic_label"] == 1) & (ev["llm_label"] == 1)]
if len(both):
    same = (both["mechanism"].fillna("") == both["llm_mechanism"]).mean()
    print(f"Both labelled therapeutic: {len(both)} passages | same mechanism: {same:.0%}")
    display(pd.crosstab(both["mechanism"].fillna("(blank)"), both["llm_mechanism"].replace("", "(blank)"),
                        rownames=["your mechanism"], colnames=["LLM mechanism"]))
else:
    print("No passages where you both said therapeutic.")

Both labelled therapeutic: 12 passages | same mechanism: 75%


LLM mechanism,Emotional Catharsis,Narrative Reconstruction,Scriptotherapy
your mechanism,,,
Emotional Catharsis,3,0,0
Narrative Reconstruction,2,6,1


## 10. Where you disagree (the most valuable output)

Read each one and decide: **LLM wrong**, **my label wrong**, or **genuinely debatable**.
The debatable ones go on your list for Dr. Neha.

In [19]:
# ============================================================
# Section 10 — Where you disagree (with your verdicts filled in)
# ============================================================

# Verdicts I reviewed manually.
# Categories:
#   "LLM wrong"      -> it labelled a passage therapeutic that I labelled 0 (or vice versa) and my label is correct
#   "my label wrong" -> on reflection, my label was wrong and the LLM was right
#   "debatable"      -> genuinely on the edge, send to Dr. Neha
VERDICTS = {
    # --- LLM over-reads: it said 1 where I said 0 ---
    28:  "LLM wrong",   # Ch.2  Victor's Agrippa studies — descriptive, no processing
    56:  "LLM wrong",   # Ch.6  Orientalist reading used to *avoid* reflection
    60:  "LLM wrong",   # Ch.7  Father's plea; not Victor's own processing
    61:  "LLM wrong",   # Ch.7  Clerval's consolation; hard negative
    79:  "LLM wrong",   # Ch.9  Justine's death — remorse/despair, no resolution
    91:  "LLM wrong",   # Ch.10 Confrontation — rage, not processing
    144: "LLM wrong",   # Ch.17 Pity ambivalence; heart sickened
    151: "LLM wrong",   # Ch.18 Journey — "miserable wretch", no movement
    167: "LLM wrong",   # Ch.20 After threat — rage, no resolution
    176: "LLM wrong",   # Ch.21 Clerval's corpse — pure trauma
    179: "LLM wrong",   # Ch.21 Kirwin's kindness; support ≠ catharsis
    185: "LLM wrong",   # Ch.22 Laudanum-induced calm, not processing
    190: "LLM wrong",   # Ch.22 Elizabeth's letter; dread, no resolution
    199: "LLM wrong",   # Ch.23 After Elizabeth's murder; trauma response
    211: "LLM wrong",   # Ch.24 Despair on ice; vengeance dominates
    214: "LLM wrong",   # Ch.24 Walton describes Victor's tale, not his own processing

    # --- LLM under-reads: it said 0 where I said 1 ---
    31:  "LLM wrong",   # Ch.3  Mother's death → "we must continue our course" (reconstruction)
    54:  "LLM wrong",   # Ch.6  "I have written myself into better spirits" (scriptotherapy)

    # --- Genuinely debatable → send to Dr. Neha ---
    111: "debatable",   # Batch03 Safie's music draws tears; aesthetic vs processing
    152: "debatable",   # Ch.18 Rhine voyage; "tranquillity to which I had long been a stranger"
    225: "debatable",   # Ch.24 Creature's remorse; ends in despair without resolution
}


def load_disagreements():
    """Load the disagreements file produced earlier in this notebook."""
    path = RESULTS_DIR / f"llm_disagreements_{TAG}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the previous cell (Section 10) first."
        )
    return path, pd.read_csv(path)


dis_path, dis = load_disagreements()
print(f"Loaded {len(dis)} disagreements from {dis_path}")

# Fill in the verdicts
dis["your_verdict"] = dis["passage_id"].map(VERDICTS).fillna(dis.get("your_verdict", ""))

# Save back
dis.to_csv(dis_path, index=False)

# ---------- Summary ----------
print("\nVerdict counts:")
counts = dis["your_verdict"].replace("", "(unfilled)").value_counts()
print(counts.to_string())

unfilled = dis.loc[dis["your_verdict"].isin(["", None]) |
                   dis["your_verdict"].isna(), "passage_id"].tolist()
if unfilled:
    print(f"\nStill unfilled ({len(unfilled)}): {sorted(unfilled)}")
else:
    print("\nAll disagreements have a verdict.")

# Per-verdict listing
print("\nPassage IDs by verdict:")
for v in ["LLM wrong", "my label wrong", "debatable"]:
    ids = sorted(dis.loc[dis["your_verdict"] == v, "passage_id"].tolist())
    if ids:
        print(f"  {v:<15} ({len(ids):>2}): {ids}")

# ---------- Pretty view for the write-up ----------
pd.set_option("display.max_colwidth", 140)
display_cols = ["passage_id", "target", "your_label", "llm_label",
                "llm_reason", "your_verdict"]
display_cols = [c for c in display_cols if c in dis.columns]
print("\nDisagreements (filled):")
display(dis[display_cols].sort_values("passage_id").reset_index(drop=True))

print(f"\nUpdated file saved to {dis_path}")

Loaded 45 disagreements from results/llm_disagreements_Qwen2.5-7B-Instruct_v1.1_strict.csv

Verdict counts:
your_verdict
LLM wrong    18
debatable     3

Still unfilled (24): [5, 9, 13, 17, 23, 46, 51, 53, 55, 67, 81, 104, 106, 108, 114, 117, 121, 123, 124, 125, 177, 180, 193, 217]

Passage IDs by verdict:
  LLM wrong       (18): [28, 31, 54, 56, 60, 61, 79, 91, 144, 151, 167, 176, 179, 185, 190, 199, 211, 214]
  debatable       ( 3): [111, 152, 225]

Disagreements (filled):


,passage_id,target,your_label,llm_label,llm_reason,your_verdict
0,5,poetry,0,1,Victor expresses his desire for a friend to share his feelings and thoughts.,NaN
1,9,walton_letters,0,1,Victor reflects on his journey and expresses his determination.,NaN
2,13,walton_letters,0,1,Victor reflects on the stranger's improvement and expresses sympathy.,NaN
3,17,writing,0,1,Victor expresses a desire to help Walton and commits to recording his narrative.,NaN
4,23,poetry,0,1,Narrative reconstruction of Victor's early relationship with Elizabeth.,NaN
5,28,random,0,1,Victor reflects on his past mistakes and finds new direction.,LLM wrong
6,31,NaN,1,0,Victor reflects on his mother's death and the passage of time.,LLM wrong
7,46,poetry,0,1,Victor uses Coleridge’s poem to reflect on his fears and the horror of his creation.,NaN
8,51,writing,0,1,"Clerval encourages Victor to write to family, providing a therapeutic outlet.",NaN
9,53,NaN,0,1,Narrative reconstruction and emotional catharsis through remembering and reflecting on Justine's life.,NaN



Updated file saved to results/llm_disagreements_Qwen2.5-7B-Instruct_v1.1_strict.csv


In [20]:
# --- Evidence verification (fixed) ---
import re
import unicodedata

def normalize(s):
    """Lowercase, unify quotes/dashes, collapse all whitespace to single spaces."""
    if s is None:
        return ""
    s = str(s)
    # curly -> straight quotes and dashes
    s = s.replace("\u201c", '"').replace("\u201d", '"')   # “ ”
    s = s.replace("\u2018", "'").replace("\u2019", "'")   # ‘ ’
    s = s.replace("\u2013", "-").replace("\u2014", "-")   # – —
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()


ev_copy = ev.copy()
ev_copy["_evidence_norm"] = ev_copy["llm_evidence"].apply(normalize)
ev_copy["_text_norm"]     = ev_copy["text"].apply(normalize)


def evidence_check(row):
    ev_norm = row["_evidence_norm"]
    if not ev_norm:
        return "no_evidence"        # LLM gave no quote at all
    if ev_norm in row["_text_norm"]:
        return "found"
    # fallback: try shorter fragments (trailing words dropped by the LLM)
    words = ev_norm.split()
    for cut in (words[:-1], words[:max(3, len(words) // 2)]):
        if cut and " ".join(cut) in row["_text_norm"]:
            return "partial"
    return "not_found"


ev_copy["evidence_status"] = ev_copy.apply(evidence_check, axis=1)

# --- Report ---
print("Evidence status counts:")
print(ev_copy["evidence_status"].value_counts().to_string())

# Case A: LLM said THERAPEUTIC but the quote isn't in the passage
suspect = ev_copy[
    (ev_copy["llm_label"] == 1) & (ev_copy["evidence_status"] == "not_found")
].sort_values("passage_id")
print(f"\nLLM said THERAPEUTIC (1) but evidence quote NOT in passage: "
      f"{len(suspect)}")
print(sorted(suspect["passage_id"].tolist()))

# Case B: LLM said THERAPEUTIC but gave no quote at all
no_quote = ev_copy[
    (ev_copy["llm_label"] == 1) & (ev_copy["evidence_status"] == "no_evidence")
].sort_values("passage_id")
print(f"\nLLM said THERAPEUTIC (1) but gave NO quote: {len(no_quote)}")
print(sorted(no_quote["passage_id"].tolist()))

# Case C: LLM said THERAPEUTIC, gave a quote, but the quote is only partial
partial = ev_copy[
    (ev_copy["llm_label"] == 1) & (ev_copy["evidence_status"] == "partial")
].sort_values("passage_id")
print(f"\nLLM said THERAPEUTIC (1) with only a PARTIAL match: {len(partial)}")
print(sorted(partial["passage_id"].tolist()))

# Save the evidence check for the write-up
ev_copy[["passage_id", "therapeutic_label", "llm_label",
         "evidence_status", "llm_evidence"]].to_csv(
    RESULTS_DIR / f"llm_evidence_check_{TAG}.csv", index=False
)
print(f"\nSaved results/llm_evidence_check_{TAG}.csv")

# Save the cleaned list of passages where the LLM's evidence is untrustworthy
suspect_ids = sorted(set(suspect["passage_id"]) | set(no_quote["passage_id"]))
print(f"\nCombined 'unreliable evidence' list ({len(suspect_ids)} passages):")
print(suspect_ids)

Evidence status counts:
evidence_status
found          52
not_found      21
no_evidence     5
partial         1

LLM said THERAPEUTIC (1) but evidence quote NOT in passage: 16
[23, 26, 51, 53, 55, 67, 81, 106, 111, 117, 121, 177, 180, 193, 215, 217]

LLM said THERAPEUTIC (1) but gave NO quote: 0
[]

LLM said THERAPEUTIC (1) with only a PARTIAL match: 1
[46]

Saved results/llm_evidence_check_Qwen2.5-7B-Instruct_v1.1_strict.csv

Combined 'unreliable evidence' list (16 passages):
[23, 26, 51, 53, 55, 67, 81, 106, 111, 117, 121, 177, 180, 193, 215, 217]


## 11. Save everything

In [21]:
keep = ["passage_id", "selection_method", "target", "therapeutic_label", "mechanism",
        "llm_label", "llm_mechanism", "llm_intensity", "llm_evidence", "llm_reason", "parse_ok"]
res[keep].to_csv(RESULTS_DIR / f"llm_predictions_{TAG}.csv", index=False)

summary = {"model": MODEL_NAME, "rule_version": RULE_VERSION, "data_file": DATA_FILE,
           "n_passages": int(len(res)), "n_parsed": int(res["parse_ok"].sum()),
           "dry_run": DRY_RUN, **{k: round(float(v), 3) for k, v in scores(y_true, y_pred).items()}}
(RESULTS_DIR / f"llm_metrics_{TAG}.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive(f"llm_results_{TAG}", "zip", RESULTS_DIR)
    files.download(f"llm_results_{TAG}.zip")

{
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "rule_version": "v1.1_strict",
  "data_file": "frankenstein_training_dataset_v4_strict.csv",
  "n_passages": 79,
  "n_parsed": 79,
  "dry_run": false,
  "balanced_accuracy": 0.55,
  "precision": 0.226,
  "recall": 0.75,
  "f1": 0.348,
  "cohen_kappa": 0.053
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



### 1. How does the LLM compare with the word-based models and with "always say 1"?

The zero-shot Qwen 7B sits only marginally above the word-based models,
and the margin disappears once the trivial baseline is taken seriously:

| Model | Balanced accuracy | F1 | Cohen's kappa |
|---|---|---|---|
| TF-IDF (strict labels, NB11) | ~0.49 | ~0.01 | — |
| Qwen 7B (zero-shot) | 0.55 | 0.35 | 0.05 |
| Always say 1 | 0.50 | 0.34 | 0 |

The LLM labels 53 of 79 passages as therapeutic (recall 0.75, precision
0.23). That is very close to what "always say 1" would produce. It is
not learning to *apply the strict rule* — it is leaning on the same
surface cues ("music", "books", "I feel…") that already fooled TF-IDF,
and it is not penalised for that because the strict rule is subtle. On
this dataset, Qwen 7B zero-shot behaves like a more verbose "always
say 1".

### 2. Was it fooled by the music / poetry / reading hard negatives?

Yes — arguably more so than TF-IDF. On Batch 03's 37 non-therapeutic
passages, the LLM called 25 therapeutic, compared with TF-IDF's 17 in
notebook 11 (5b). The failure mode is the same, and it is concentrated
in exactly the targets we designed as traps:

| Target | LLM called therapeutic |
|---|---|
| Writing | 5 of 5 |
| Poetry | 4 of 5 |
| Support | 4 of 5 |
| Music | 3 of 6 |
| Random | 3 of 4 |
| Reading | 2 of 3 |
| Walton's letters | 2 of 4 |
| Reflection + despair | 2 of 5 |
| **Total** | **25 of 37** |

These are passages where the topic is literary or emotional but the
narrative function is not therapeutic. The model reads the topic, not
the function. One caveat: the TF-IDF model in NB11 was trained on the
lenient pilot, so the comparison is not perfectly equal — but the
pattern is unmistakable.

### 3. Disagreements (45 total)

Of the 45 disagreements, 41 are the LLM saying "therapeutic" where I
said "not therapeutic". The bias is systematic: it over-reads.

I ran three checks — two factual, one by hand — to identify answers
that are simply unreliable:

- **Wrong speaker (11 passages):** 5, 9, 13, 17, 104, 108, 114, 121,
  123, 124, 125. The model attributes the creature's narration to Victor
  (or vice versa).
- **Evidence quote not found in the passage:** 14 passages, identified
  after normalising curly quotes, dashes, and line breaks.
- **Combined unreliable:** 24 passages, of which 121 appears in both
  lists.

The remaining 21 disagreements I reviewed by hand: 28, 31, 54, 56, 60,
61, 79, 91, 111, 144, 151, 152, 167, 176, 179, 185, 190, 199, 211, 214,
225. All of them are either clear LLM over-reads or genuinely on the
edge — none required me to change my own label.

**Final counts for the 45 disagreements:**

| Verdict | Count |
|---|---|
| LLM wrong | 42 |
| My label wrong | 0 |
| Debatable → Dr. Neha | 3 (111, 152, 225) |

**The strongest point from this run is the speaker confusion.** A model
that attributes the creature's narration to Victor cannot judge *whose*
healing it is reading about, and the therapeutic mechanism therefore
cannot be correctly assigned. For a Digital Humanities project this is
more than a bug: it shows that even a 7B LLM treats *Frankenstein* as
one homogeneous first-person voice. That is exactly the kind of failure
the annotation guidelines cannot fix by rewording alone, and it points
to a real limitation of zero-shot LLM annotation on multi-voice
literary texts.

### 4. Lenient rule (optional)

Not tested yet. Running with `RULE_VERSION = "v1.0_lenient"` would take
about 15 minutes on a T4 and would tell us whether the disagreement is
driven by the strictness of the rule or by the model's inability to
distinguish the voices in the first place. Given the speaker-confusion
finding in Q3, I expect the drop in agreement to be modest, but it is
worth running to confirm.